
# Caderno de Trading — LULU, GAP (GPS), ANF — Yahoo Finance  
**ADX/ATR, filtros EMA50/200, Alarmes + Notificações — (FIX Candles mplfinance)**

*Data:* **2025-10-04**  
**Correção:** compatibilidade com versões antigas do `mplfinance` que exigem *Python floats* e índice sem timezone.


## 0) Dependências

In [ ]:

import sys, subprocess
def _pip_install(pkg: str):
    try:
        __import__(pkg.split("==")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])
for p in ["yfinance>=0.2.40","pandas>=2.0.0","matplotlib>=3.7.0","mplfinance>=0.12.10b0","pytz","numpy>=1.24.0","requests>=2.31.0"]:
    _pip_install(p)
print("Dependências OK.")


## 1) Utilitários (inclui **to_mpf_compat** para candles)

In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt, yfinance as yf, mplfinance as mpf, pytz, json, csv, hashlib, smtplib, requests
from email.message import EmailMessage
from datetime import datetime, timedelta
from pathlib import Path

def to_mpf_compat(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["Open","High","Low","Close","Volume"]
    out = df.copy()
    out = out[cols].apply(pd.to_numeric, errors="coerce").dropna()
    for c in cols:
        out[c] = [float(x) if pd.notnull(x) else float("nan") for x in out[c].tolist()]
    if isinstance(out.index, pd.DatetimeIndex) and out.index.tz is not None:
        out.index = out.index.tz_localize(None)
    return out

def plot_candles(df: pd.DataFrame, titulo: str = ""):
    dfm = to_mpf_compat(df)
    mpf.plot(dfm, type="candle", style="yahoo", title=titulo, volume=False)



## 2) Teste rápido (use seu DataFrame `ult_24h` aqui)
Se você estiver usando o caderno completo anterior, **substitua** a função `plot_candles` por esta e adicione `to_mpf_compat`.  
A seguir, um pequeno teste isolado:


In [ ]:

LOCAL_TZ = "America/Sao_Paulo"
tk = "LULU"
dados_2d_1h = yf.download(tk, period="2d", interval="60m", auto_adjust=False, progress=False)
if not dados_2d_1h.index.tz:
    dados_2d_1h.index = dados_2d_1h.index.tz_localize("UTC")
dados_2d_1h_local = dados_2d_1h.tz_convert(LOCAL_TZ)
corte = dados_2d_1h_local.index.max() - pd.Timedelta("24H")
ult_24h = dados_2d_1h_local.loc[dados_2d_1h_local.index > corte].copy()

plot_candles(ult_24h, titulo=f"{tk} — Candles 1h (últimas 24h) [TZ: {LOCAL_TZ}]")
print("OK: candles plotados com compatibilidade antiga do mplfinance.")
